# 03a — Assemble the audited analysis dataset

Performs validated one-to-one merging and creates explicit measurement, diagnosis, segmentation, and clinical gates.

This notebook saves its visual summary and audit tables into separate `figures/` and `tables/` directories. Its final cell states the main output, any decision required, and whether the next stage is allowed.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from IPython.display import Image, Markdown, display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the paper_1 project.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
MAIN_OUTPUTS = ROOT / "MAIN outputs"
MAIN_OUTPUTS.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def read_table(path_without_suffix):
    stem = Path(path_without_suffix)
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing table: {parquet} or {csv}")

def stage_directories(relative_stage):
    stage = OUTPUT / relative_stage
    figures = stage / "figures"
    tables = stage / "tables"
    figures.mkdir(parents=True, exist_ok=True)
    tables.mkdir(parents=True, exist_ok=True)
    return stage, figures, tables

def save_table(frame, directory, name):
    path = Path(directory) / f"{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, directory, name):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    png = directory / f"{name}.png"
    svg = directory / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

def stage_gate(stage_name, can_continue, reasons, next_step):
    status = "PASS — safe to continue" if can_continue else "BLOCKED — decision/action required"
    color = "#1B7F3A" if can_continue else "#B22222"
    details = "\n".join(f"- {reason}" for reason in reasons) if reasons else "- No blocking findings."
    display(Markdown(
        f"### {stage_name}: <span style='color:{color}'>{status}</span>\n\n"
        f"{details}\n\n**Next step:** {next_step}"
    ))
    return can_continue

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("MAIN outputs:", MAIN_OUTPUTS)


## Main output and decisions

Main output: `outputs/03_dataset_assembly/paper1_analysis_dataset.csv`. Investigate any merge/extraction failure or unexpected task-completion missingness before analysis.

In [ ]:
RUN_ASSEMBLY = False  # Change to True only when you intend to run this stage.

if RUN_ASSEMBLY:
    run_cli('assemble')
else:
    print('Assembly not run. Enable only after all six feature-family gates pass.')

In [ ]:
STAGE, FIGURES, TABLES = stage_directories("03_dataset_assembly")
flow = read_table(STAGE / "eligibility_flow_counts")
data = read_table(STAGE / "paper1_analysis_dataset")
save_table(flow, TABLES, "eligibility_flow_counts")
display(flow)

plot = flow.sort_values("n_true")
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.barplot(data=plot, x="n_true", y="criterion", color="#4C78A8", ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=3)
ax.set(title="Dataset assembly eligibility and exclusion gates", xlabel="Logical recordings", ylabel="")
fig.tight_layout()
save_figure(fig, FIGURES, "eligibility_flow")
plt.show()

blocking_columns = ["feature_extraction_missing"]
blocking = [
    f"{column}: {int(data[column].fillna(False).sum())} recordings"
    for column in blocking_columns
    if column in data and data[column].fillna(False).any()
]
assembly_ready = stage_gate(
    "Dataset assembly",
    not blocking,
    blocking,
    "Open the dataset statistics notebook if PASS; otherwise reconcile the saved rows.",
)